In [0]:
from pyspark.sql import functions as F

In [0]:
# STEP 1: CREATE SAMPLE RAW DATA

data = [
    (101, "Laptop", 2, 50000),
    (102, "Phone", 5, 25000),
    (103, "Tablet", 3, 15000),
    (104, "Headphones", 4, 2000),
    (105, "Keyboard", 3, 1500),
    (106, "Laptop", 1, 50000),
    (107, "Mouse", 5, 800),
    (108, "Monitor", 2, 12000)
]

columns = ["Order_ID", "Product", "Quantity", "Price"]

raw_df = spark.createDataFrame(data, columns)

print("STEP 1: RAW SALES DATA")
display(raw_df)

STEP 1: RAW SALES DATA


Order_ID,Product,Quantity,Price
101,Laptop,2,50000
102,Phone,5,25000
103,Tablet,3,15000
104,Headphones,4,2000
105,Keyboard,3,1500
106,Laptop,1,50000
107,Mouse,5,800
108,Monitor,2,12000


In [0]:
# STEP 2: CHECK DATA QUALITY

print("STEP 2: CHECK FOR MISSING VALUES")

raw_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in raw_df.columns
]).show()


STEP 2: CHECK FOR MISSING VALUES
+--------+-------+--------+-----+
|Order_ID|Product|Quantity|Price|
+--------+-------+--------+-----+
|       0|      0|       0|    0|
+--------+-------+--------+-----+



In [0]:
# STEP 3: REMOVE DUPLICATE RECORDS

clean_df = raw_df.dropDuplicates(["Order_ID"])

print("STEP 3: DATA AFTER DUPLICATE CHECK")
display(clean_df)

STEP 3: DATA AFTER DUPLICATE CHECK


Order_ID,Product,Quantity,Price
101,Laptop,2,50000
102,Phone,5,25000
103,Tablet,3,15000
104,Headphones,4,2000
105,Keyboard,3,1500
106,Laptop,1,50000
107,Mouse,5,800
108,Monitor,2,12000


In [0]:
# STEP 4: VALIDATE AND CLEAN DATA

clean_df = clean_df.filter(
    (F.col("Order_ID").isNotNull()) &
    (F.col("Product").isNotNull()) &
    (F.col("Quantity") > 0) &
    (F.col("Price") >= 0)
)

print("STEP 4: CLEANED SALES DATA")
display(clean_df)

STEP 4: CLEANED SALES DATA


Order_ID,Product,Quantity,Price
101,Laptop,2,50000
102,Phone,5,25000
103,Tablet,3,15000
104,Headphones,4,2000
105,Keyboard,3,1500
106,Laptop,1,50000
107,Mouse,5,800
108,Monitor,2,12000


In [0]:
# STEP 5: TRANSFORM DATA

sales_df = clean_df.withColumn(
    "Revenue",
    F.col("Quantity") * F.col("Price")
)

print("STEP 5: SALES DATA WITH REVENUE")
display(sales_df)

STEP 5: SALES DATA WITH REVENUE


Order_ID,Product,Quantity,Price,Revenue
101,Laptop,2,50000,100000
102,Phone,5,25000,125000
103,Tablet,3,15000,45000
104,Headphones,4,2000,8000
105,Keyboard,3,1500,4500
106,Laptop,1,50000,50000
107,Mouse,5,800,4000
108,Monitor,2,12000,24000


In [0]:
# STEP 6: ANALYZE TOTAL REVENUE

total_revenue = sales_df.agg(
    F.sum("Revenue").alias("Total_Revenue")
)

print("STEP 6: TOTAL REVENUE")
display(total_revenue)

STEP 6: TOTAL REVENUE


Total_Revenue
360500


In [0]:
# STEP 7: REVENUE BY PRODUCT

product_revenue = sales_df.groupBy("Product").agg(
    F.sum("Revenue").alias("Total_Revenue"),
    F.sum("Quantity").alias("Total_Quantity_Sold")
).orderBy(F.desc("Total_Revenue"))

print("STEP 7: REVENUE BY PRODUCT")
display(product_revenue)

STEP 7: REVENUE BY PRODUCT


Product,Total_Revenue,Total_Quantity_Sold
Laptop,150000,3
Phone,125000,5
Tablet,45000,3
Monitor,24000,2
Headphones,8000,4
Keyboard,4500,3
Mouse,4000,5


In [0]:
# STEP 8: GENERATE BUSINESS INSIGHTS

print("STEP 8: TOP-SELLING PRODUCTS BY REVENUE")
display(product_revenue.limit(5))

print("PROJECT COMPLETED SUCCESSFULLY!")

STEP 8: TOP-SELLING PRODUCTS BY REVENUE


Product,Total_Revenue,Total_Quantity_Sold
Laptop,150000,3
Phone,125000,5
Tablet,45000,3
Monitor,24000,2
Headphones,8000,4


PROJECT COMPLETED SUCCESSFULLY!
